In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python

import numpy as np
import pandas as pd
import os
import sys

print("Python environment initialized.")


Python environment initialized.


# QSED HPatches Benchmark — Entropy Plus Euclidean Augmented Vectors

This notebook evaluates the **Quantum-Inspired Local Von Neumann Entropy Feature Descriptor** on the **HPatches** benchmark and investigates the construction and validation of **Entropy + Euclidean Augmented Descriptors**:

$$g_{ij, k} = \sqrt{\left| (F^A_{i, k})^2 - (F^B_{j, k})^2 \right|}$$
$$C_{ij} = \|g_{ij}\|_2 = \sqrt{\sum_{k=0}^{D-1} \left| (F^A_{i, k})^2 - (F^B_{j, k})^2 \right|}$$

### Key Properties & Protocol:
- **Raw Von Neumann Entropy Bits**: Descriptors are extracted with `norm="none"` (unnormalized entropy bits).
- **Configurations Evaluated**: $(n, y) \in \{(7, 3), (7, 5), (9, 3), (9, 5)\}$, where descriptor dimension $D = n^2 - 1$.
- **Augmented Vector Construction**:
  - For each reference keypoint $i$, appends its score-matrix row $C[i, :]$ to its $D$-entropy descriptor.
  - For each target keypoint $j$, appends the corresponding score-matrix column $C[:, j]$ to its $D$-entropy descriptor.
  - Euclidean portions are zero-padded to 1,024 values, producing augmented vectors of shape `(num_keypoints, D + 1024)`.
  - An in-memory boolean validity mask retains which elements are true scores vs zero-padding, ensuring true 0.0 scores are distinguishable from padding zeros.
- **Diagnostics & Inspection Only**: Augmented vectors are inspected and logged for shapes, valid lengths, and zero distinguishability; they are **NOT** used for matching, RANSAC, or benchmark metrics, and are discarded after each pair (never persisted).
- **Matching & Homography Estimation**: Evaluated using the literal score matrix $C$ alone across all 116 sequences for all configurations.
- **HPatches Dataset Only**: No Oxford Affine or MegaDepth benchmarks.


In [2]:
# Clone or update repository in Kaggle environment, or navigate to repo root
import os, sys

if os.path.exists("/kaggle/working"):
    !git clone https://github.com/yaswanthvuppala/24-D-VonNeumon-Similarity-Q.git /kaggle/working/24-D-VonNeumon-Similarity-Q 2>/dev/null || (cd /kaggle/working/24-D-VonNeumon-Similarity-Q && git pull origin main)
    %cd /kaggle/working/24-D-VonNeumon-Similarity-Q
    !pip install -q -r requirements.txt
else:
    repo_dir = os.path.abspath(".")
    if not os.path.isdir(os.path.join(repo_dir, "src")):
        candidate = os.path.abspath(os.path.join(repo_dir, "24-D-VonNeumon-Similarity-Q"))
        if os.path.isdir(os.path.join(candidate, "src")):
            %cd {candidate}
            repo_dir = candidate
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
    print(f"Working directory: {os.getcwd()}")


/kaggle/working/24-D-VonNeumon-Similarity-Q


In [3]:
import os

# Discover or download HPatches sequences dataset ONLY (no Oxford, no MegaDepth)
dataset_dir = "data/hpatches-sequences-release"

if not os.path.isdir(dataset_dir) and os.path.isdir("/kaggle/input"):
    for root, dirs, _ in os.walk("/kaggle/input"):
        if "hpatches-sequences-release" in dirs:
            candidate = os.path.join(root, "hpatches-sequences-release")
            if os.path.isdir(candidate) and os.path.isfile(os.path.join(candidate, "v_weapons", "1.ppm")):
                dataset_dir = candidate
                print(f"Found mounted HPatches dataset at: {dataset_dir}")
                break

if not os.path.isdir(dataset_dir):
    os.makedirs("data", exist_ok=True)
    print("Downloading HPatches sequences (~1.28 GB zip)...")
    !wget -c -O data/hpatches-sequences-release.zip "https://huggingface.co/datasets/vbalnt/hpatches/resolve/main/hpatches-sequences-release.zip"
    !unzip -q data/hpatches-sequences-release.zip -d data/
    !rm -f data/hpatches-sequences-release.zip
    dataset_dir = "data/hpatches-sequences-release"

if os.path.isdir(dataset_dir):
    extracted = [d for d in os.listdir(dataset_dir) if os.path.isdir(os.path.join(dataset_dir, d))]
    print(f"HPatches dataset ready: {len(extracted)} sequences found in '{dataset_dir}'.")
else:
    print(f"Dataset directory '{dataset_dir}' not yet populated.")


--2026-10-30 18:43:22--  https://huggingface.co/datasets/vbalnt/hpatches/resolve/main/hpatches-sequences-release.zip
Resolving huggingface.co (huggingface.co)... 13.226.251.20, 13.226.251.112, 13.226.251.66, ...
Connecting to huggingface.co (huggingface.co)|13.226.251.20|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/67e3e06b388d76021c8bfec0/9df11ca526095d0577e2b6ea0aabd16a981b831880766935812204cc8fe5e051?response-content-type=application%2Fzip&X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27hpatches-sequences-release.zip%3B+filename%3D%22hpatches-sequences-release.zip%22%3B&xip=dNm77CvppGI&user_id=public&Expires=1793389402&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjdlM2UwNmIzODhkNzYwMjFjOGJmZWMwLzlkZjExY2E1MjYwOTVkMDU3N2UyYjZlYTBhYWJkMTZhOTgxYjgzMTg4MDc2NjkzNTgxMjIwNGNjOGZlNWUwNTFcXD9yZXNwb25zZS1jb250ZW50LXR5cGU9KiZYLVhldC1DYXM

In [4]:
# =============================================================================
# SANITY CELL: LITERAL FORMULA, AUGMENTED VECTORS, PADDING & VALIDITY MASKS
# =============================================================================
import numpy as np

print("Running sanity checks for augmented vectors, padding, and validity masks...")

# 1. Negative raw radicand test
fa_val, fb_val = 1.5, 2.5
radicand = (fa_val ** 2) - (fb_val ** 2)  # 2.25 - 6.25 = -4.0 < 0
assert radicand < 0
g_val = np.sqrt(np.abs(radicand))
assert np.isclose(g_val, 2.0)
print(f"[Sanity 1/4 PASSED] Negative raw radicand ({radicand}) produces real g = {g_val}")

# 2. Build sample descriptors and score matrix
np.random.seed(42)
N_A, N_B, D = 15, 20, 48
desc_a = np.random.uniform(0.1, 3.1, (N_A, D))
desc_b = np.random.uniform(0.1, 3.1, (N_B, D))

fa_sq = np.square(desc_a)[:, np.newaxis, :]
fb_sq = np.square(desc_b)[np.newaxis, :, :]
abs_diff = np.abs(fa_sq - fb_sq)
C = np.sqrt(np.sum(abs_diff, axis=2))
assert C.shape == (N_A, N_B)
assert np.all(np.isfinite(C))
print(f"[Sanity 2/4 PASSED] Score matrix C of shape {C.shape} computed and verified finite.")

# 3. Construct Augmented Vectors with 1,024-padding and validity masks
MAX_PAD = 1024

# Reference: append score rows C[i, :], pad to 1024
euclid_ref_padded = np.zeros((N_A, MAX_PAD), dtype=np.float64)
mask_ref = np.zeros((N_A, MAX_PAD), dtype=bool)
euclid_ref_padded[:, :N_B] = C
mask_ref[:, :N_B] = True
aug_ref = np.hstack([desc_a, euclid_ref_padded])

# Target: append score cols C[:, j], pad to 1024
euclid_tgt_padded = np.zeros((N_B, MAX_PAD), dtype=np.float64)
mask_tgt = np.zeros((N_B, MAX_PAD), dtype=bool)
euclid_tgt_padded[:, :N_A] = C.T
mask_tgt[:, :N_A] = True
aug_tgt = np.hstack([desc_b, euclid_tgt_padded])

assert aug_ref.shape == (N_A, D + MAX_PAD), f"Ref shape {aug_ref.shape} != {(N_A, D + MAX_PAD)}"
assert aug_tgt.shape == (N_B, D + MAX_PAD), f"Tgt shape {aug_tgt.shape} != {(N_B, D + MAX_PAD)}"
assert np.all(aug_ref[:, :D] == desc_a), "Reference entropy portion corrupted!"
assert np.all(aug_tgt[:, :D] == desc_b), "Target entropy portion corrupted!"
assert np.all(aug_ref[:, D:D+N_B] == C), "Reference Euclidean portion mismatch!"
assert np.all(aug_tgt[:, D:D+N_A] == C.T), "Target Euclidean portion mismatch!"
assert np.all(aug_ref[:, D+N_B:] == 0.0), "Reference padding non-zero!"
assert np.all(aug_tgt[:, D+N_A:] == 0.0), "Target padding non-zero!"
print(f"[Sanity 3/4 PASSED] Augmented shapes: ref {aug_ref.shape}, tgt {aug_tgt.shape}. Padding is exact 0.0.")

# 4. Validity Mask and Distinguishability Check
# Inject a synthetic exact-match score of 0.0 into C[2, 3]
C_with_zero = C.copy()
C_with_zero[2, 3] = 0.0
euclid_test = np.zeros((N_A, MAX_PAD), dtype=np.float64)
mask_test = np.zeros((N_A, MAX_PAD), dtype=bool)
euclid_test[:, :N_B] = C_with_zero
mask_test[:, :N_B] = True

# In euclid_test[2], index 3 has value 0.0 (real match) and mask=True
# Index N_B has value 0.0 (padded zero) and mask=False
assert euclid_test[2, 3] == 0.0 and mask_test[2, 3] == True, "Failed to identify real zero score!"
assert euclid_test[2, N_B] == 0.0 and mask_test[2, N_B] == False, "Failed to identify padded zero!"
print(f"[Sanity 4/4 PASSED] Validity mask cleanly distinguishes real zero scores from zero-padding.")


Running sanity checks for augmented vectors, padding, and validity masks...
[Sanity 1/4 PASSED] Negative raw radicand (-4.0) produces real g = 2.0
[Sanity 2/4 PASSED] Score matrix C of shape (15, 20) computed and verified finite.
[Sanity 3/4 PASSED] Augmented shapes: ref (15, 1072), tgt (20, 1072). Padding is exact 0.0.
[Sanity 4/4 PASSED] Validity mask cleanly distinguishes real zero scores from zero-padding.


In [5]:
# =============================================================================
# NOTEBOOK-LOCAL EVALUATION PIPELINE: ENTROPY PLUS EUCLIDEAN AUGMENTED
# =============================================================================
import os
import csv
import time
from datetime import datetime
from typing import List, Dict, Tuple, Optional, Any
import numpy as np
from tqdm import tqdm

from src.feature.entropy_descriptor import (
    extract_all_entropy_descriptors,
    detect_default_keypoints,
)
from src.feature.hpatches_adapter import (
    load_hpatches_image,
    load_homography,
    scale_homography,
    estimate_homography_ransac,
    compute_corner_error,
    compute_homography_auc,
    get_hpatches_sequences,
    get_sequence_pairs,
    format_ripe_comparison_table,
)


def compute_literal_score_matrix(desc_a: np.ndarray, desc_b: np.ndarray) -> np.ndarray:
    """
    Computes candidate keypoint-pair score matrix C using the literal formula:
        g_{ij, k} = sqrt(|(F^A_{i, k})^2 - (F^B_{j, k})^2|)
        C_{ij}    = ||g_{ij}||_2 = sqrt(sum_k |(F^A_{i, k})^2 - (F^B_{j, k})^2|)
    """
    n_a = desc_a.shape[0]
    n_b = desc_b.shape[0]
    if n_a == 0 or n_b == 0:
        return np.empty((n_a, n_b), dtype=np.float64)

    fa_sq = np.square(desc_a)[:, np.newaxis, :]
    fb_sq = np.square(desc_b)[np.newaxis, :, :]
    abs_diff = np.abs(fa_sq - fb_sq)
    C = np.sqrt(np.sum(abs_diff, axis=2))
    return C


def build_augmented_vectors(
    desc_ref: np.ndarray,
    desc_tgt: np.ndarray,
    score_matrix: np.ndarray,
    max_pad: int = 1024
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Constructs augmented vectors of shape (num_keypoints, D + 1024) and boolean validity masks.
    
    For each reference keypoint i:
        appends score row C[i, :] (length N_B) padded to max_pad with zeros.
    For each target keypoint j:
        appends score col C[:, j] (length N_A) padded to max_pad with zeros.
        
    Returns:
        aug_ref: (N_A, D + 1024)
        mask_ref: (N_A, 1024)
        aug_tgt: (N_B, D + 1024)
        mask_tgt: (N_B, 1024)
    """
    n_a, d_a = desc_ref.shape
    n_b, d_b = desc_tgt.shape
    assert d_a == d_b, f"Descriptor dimensions must match: {d_a} != {d_b}"

    # Reference augmentation
    euclid_ref_padded = np.zeros((n_a, max_pad), dtype=np.float64)
    mask_ref = np.zeros((n_a, max_pad), dtype=bool)
    if n_b > 0:
        actual_nb = min(n_b, max_pad)
        euclid_ref_padded[:, :actual_nb] = score_matrix[:, :actual_nb]
        mask_ref[:, :actual_nb] = True
    aug_ref = np.hstack([desc_ref, euclid_ref_padded])

    # Target augmentation
    euclid_tgt_padded = np.zeros((n_b, max_pad), dtype=np.float64)
    mask_tgt = np.zeros((n_b, max_pad), dtype=bool)
    if n_a > 0:
        actual_na = min(n_a, max_pad)
        euclid_tgt_padded[:, :actual_na] = score_matrix.T[:, :actual_na]
        mask_tgt[:, :actual_na] = True
    aug_tgt = np.hstack([desc_tgt, euclid_tgt_padded])

    return aug_ref, mask_ref, aug_tgt, mask_tgt


def match_mutual_nn_from_scores(
    score_matrix: np.ndarray,
    max_distance: Optional[float] = None
) -> List[Tuple[int, int, float]]:
    """
    Mutual Nearest Neighbor (MNN) matching based on score matrix C alone.
    Keypoint i in A matches keypoint j in B iff:
        argmin_j C[i, j] == j  AND  argmin_i C[i, j] == i
    """
    n_a, n_b = score_matrix.shape
    if n_a == 0 or n_b == 0:
        return []

    nn_a_to_b = np.argmin(score_matrix, axis=1)
    nn_b_to_a = np.argmin(score_matrix, axis=0)

    matches = []
    for i in range(n_a):
        j = int(nn_a_to_b[i])
        if nn_b_to_a[j] == i:
            dist = float(score_matrix[i, j])
            if max_distance is None or dist <= max_distance:
                matches.append((i, j, dist))
    return matches


def run_hpatches_configuration_entropy_plus_euclidean(
    dataset_dir: str,
    output_dir: str,
    n: int,
    y: int,
    max_keypoints: int = 1024,
    max_side: int = 480,
    ransac_thresh: float = 3.0
) -> Dict[str, Any]:
    """
    Evaluates one (n, y) configuration across all HPatches sequences:
    - Generates and verifies augmented vectors (num_keypoints, D + 1024) for every pair.
    - Inspects/reports shapes and valid lengths; discards large vectors immediately.
    - Matches using score matrix C alone for RANSAC homography estimation.
    """
    os.makedirs(output_dir, exist_ok=True)
    D = n * n - 1
    AUG_DIM = D + 1024

    print("=" * 75)
    print(f"  QSED HPatches Evaluation — Entropy + Euclidean (n={n}, y={y}, D={D})")
    print(f"  Augmented Vector Width : {AUG_DIM} ({D} entropy + 1024 Euclidean)")
    print("=" * 75)
    print(f"  Normalization          : raw Von Neumann entropy bits (none)")
    print(f"  Matching               : Mutual Nearest Neighbor on literal score matrix C")
    print(f"  Augmented Vector Action: shape & validity inspection ONLY (not used for matching/RANSAC)")
    print(f"  Max Keypoints          : {max_keypoints}")
    print(f"  Max Image Side         : {max_side}px")
    print(f"  RANSAC Threshold       : {ransac_thresh}px")
    print(f"  Dataset Directory      : {dataset_dir}")
    print(f"  Output Directory       : {output_dir}")
    print("=" * 75)

    sequences = get_hpatches_sequences(dataset_dir)
    n_illum = sum(1 for s in sequences if s["type"] == "illumination")
    n_view = sum(1 for s in sequences if s["type"] == "viewpoint")
    print(f"  Found {len(sequences)} sequences ({n_illum} illumination, {n_view} viewpoint)")
    print(f"  Total image pairs : {len(sequences) * 5}\n")

    errors = {"illumination": [], "viewpoint": [], "all": []}
    match_counts = {"illumination": [], "viewpoint": [], "all": []}
    failed_homographies = {"illumination": 0, "viewpoint": 0, "all": 0}
    per_pair_records = []

    # Augmented vector diagnostic trackers
    aug_diagnostics = {
        "verified_ref_shapes": 0,
        "verified_tgt_shapes": 0,
        "verified_padding_zeros": 0,
        "verified_masks": 0,
        "ref_kps_counts": [],
        "tgt_kps_counts": [],
    }

    start_time = time.time()

    for seq_info in tqdm(sequences, desc=f"Eval n={n},y={y}", unit="seq"):
        seq_name = seq_info["name"]
        seq_path = seq_info["path"]
        seq_type = seq_info["type"]

        ref_path = os.path.join(seq_path, "1.ppm")
        img_ref, scale_ref = load_hpatches_image(ref_path, max_side=max_side)
        kps_ref = detect_default_keypoints(img_ref, max_keypoints=max_keypoints)

        if len(kps_ref) == 0:
            for pair_info in get_sequence_pairs(seq_path):
                per_pair_records.append({
                    "sequence": seq_name, "type": seq_type, "pair": pair_info["pair_name"],
                    "num_kps_ref": 0, "num_kps_tgt": 0, "num_matches": 0, "corner_error": float('inf'),
                })
                errors[seq_type].append(float('inf'))
                errors["all"].append(float('inf'))
                failed_homographies[seq_type] += 1
                failed_homographies["all"] += 1
            continue

        desc_ref = extract_all_entropy_descriptors(
            img_ref, kps_ref, ordering="raster", norm="none", n=n, y=y
        )

        for pair_info in get_sequence_pairs(seq_path):
            img_tgt, scale_tgt = load_hpatches_image(pair_info["tgt_path"], max_side=max_side)
            H_gt = load_homography(pair_info["H_path"])
            H_gt_scaled = scale_homography(H_gt, scale_ref, scale_tgt)

            kps_tgt = detect_default_keypoints(img_tgt, max_keypoints=max_keypoints)
            desc_tgt = extract_all_entropy_descriptors(
                img_tgt, kps_tgt, ordering="raster", norm="none", n=n, y=y
            )

            # Build all-keypoint score matrix C using literal formula
            C = compute_literal_score_matrix(desc_ref, desc_tgt)

            # Build Augmented Vectors of shape (num_kps, D + 1024)
            aug_ref, mask_ref, aug_tgt, mask_tgt = build_augmented_vectors(
                desc_ref, desc_tgt, C, max_pad=1024
            )

            # -------------------------------------------------------------
            # Inspect/report shapes and validity masks only
            # -------------------------------------------------------------
            assert aug_ref.shape == (len(kps_ref), AUG_DIM), f"Aug ref shape mismatch: {aug_ref.shape}"
            assert aug_tgt.shape == (len(kps_tgt), AUG_DIM), f"Aug tgt shape mismatch: {aug_tgt.shape}"
            aug_diagnostics["verified_ref_shapes"] += 1
            aug_diagnostics["verified_tgt_shapes"] += 1

            # Verify padding is strictly zeros
            assert np.all(aug_ref[:, D + len(kps_tgt):] == 0.0), "Aug ref padding non-zero!"
            assert np.all(aug_tgt[:, D + len(kps_ref):] == 0.0), "Aug tgt padding non-zero!"
            aug_diagnostics["verified_padding_zeros"] += 1

            # Verify validity masks
            assert np.all(mask_ref[:, :len(kps_tgt)] == True) and np.all(mask_ref[:, len(kps_tgt):] == False)
            assert np.all(mask_tgt[:, :len(kps_ref)] == True) and np.all(mask_tgt[:, len(kps_ref):] == False)
            aug_diagnostics["verified_masks"] += 1
            aug_diagnostics["ref_kps_counts"].append(len(kps_ref))
            aug_diagnostics["tgt_kps_counts"].append(len(kps_tgt))

            # DISCARD augmented vectors immediately (do not use for matching/RANSAC, do not persist)
            del aug_ref, mask_ref, aug_tgt, mask_tgt

            # -------------------------------------------------------------
            # Matching & RANSAC via score matrix C alone
            # -------------------------------------------------------------
            matches = match_mutual_nn_from_scores(C)
            num_matches = len(matches)

            # Discard score matrix after matching
            del C

            H_est = estimate_homography_ransac(
                kps_ref, kps_tgt, matches, ransac_thresh=ransac_thresh
            )

            if H_est is not None:
                err = compute_corner_error(H_gt_scaled, H_est, img_ref.shape)
            else:
                err = float('inf')
                failed_homographies[seq_type] += 1
                failed_homographies["all"] += 1

            errors[seq_type].append(err)
            errors["all"].append(err)
            match_counts[seq_type].append(num_matches)
            match_counts["all"].append(num_matches)

            per_pair_records.append({
                "sequence": seq_name,
                "type": seq_type,
                "pair": pair_info["pair_name"],
                "num_kps_ref": len(kps_ref),
                "num_kps_tgt": len(kps_tgt),
                "num_matches": num_matches,
                "corner_error": err,
            })

    elapsed = time.time() - start_time

    # Report generation
    results_text_lines = []
    results_text_lines.append("=" * 75)
    results_text_lines.append(f"  QSED HPatches Benchmark Results (Entropy + Euclidean Augmented)")
    results_text_lines.append("=" * 75)
    results_text_lines.append(f"  Date             : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    results_text_lines.append(f"  Configuration    : n={n}, y={y} (D={D}, Augmented Width={AUG_DIM})")
    results_text_lines.append(f"  Normalization    : raw Von Neumann entropy bits (none)")
    results_text_lines.append(f"  Matching         : Mutual Nearest Neighbor on literal score matrix C")
    results_text_lines.append(f"  Augmented Vector : shape (N, {AUG_DIM}) inspected and verified")
    results_text_lines.append(f"  Max Keypoints    : {max_keypoints}")
    results_text_lines.append(f"  Max Image Side   : {max_side}px")
    results_text_lines.append(f"  RANSAC Threshold : {ransac_thresh}px")
    results_text_lines.append(f"  Total Time       : {elapsed:.1f}s")
    results_text_lines.append("")

    # Add augmented vector diagnostics
    results_text_lines.append("  Augmented Vector Inspection Summary:")
    results_text_lines.append(f"    Total Pairs Inspected         : {aug_diagnostics['verified_ref_shapes']}")
    results_text_lines.append(f"    Augmented Vector Dimensions   : (N_ref, {AUG_DIM}) and (N_tgt, {AUG_DIM}) [100% verified]")
    results_text_lines.append(f"    Zero-Padding Integrity        : 100% verified exact zeros")
    results_text_lines.append(f"    Validity Mask Integrity       : 100% verified clean distinction")
    ref_kps = np.array(aug_diagnostics["ref_kps_counts"])
    tgt_kps = np.array(aug_diagnostics["tgt_kps_counts"])
    results_text_lines.append(f"    Ref Keypoints (valid lengths) : min={np.min(ref_kps)}, mean={np.mean(ref_kps):.1f}, max={np.max(ref_kps)}")
    results_text_lines.append(f"    Tgt Keypoints (valid lengths) : min={np.min(tgt_kps)}, mean={np.mean(tgt_kps):.1f}, max={np.max(tgt_kps)}")
    results_text_lines.append("")

    summary_metrics = {}
    for cat in ["illumination", "viewpoint", "all"]:
        cat_errs = errors[cat]
        cat_matches = match_counts[cat]
        cat_failed = failed_homographies[cat]
        if len(cat_errs) == 0:
            continue
        auc = compute_homography_auc(cat_errs)
        summary_metrics[cat] = {"auc": auc, "failed": cat_failed, "total": len(cat_errs)}

        header = f"\n  {cat.upper()} ({len(cat_errs)} pairs)"
        results_text_lines.append(header)
        results_text_lines.append("-" * 55)
        for key, val in auc.items():
            results_text_lines.append(f"    {key:>10s} : {val:6.2f}%")
        if len(cat_matches) > 0:
            mc = np.array(cat_matches)
            results_text_lines.append(
                f"    Matches: mean={np.mean(mc):.1f}, median={np.median(mc):.0f}, min={np.min(mc)}, max={np.max(mc)}"
            )
        results_text_lines.append(f"    Failed homographies: {cat_failed}/{len(cat_errs)}")

    all_auc = summary_metrics.get("all", {}).get("auc", {})
    ripe_table = format_ripe_comparison_table(
        our_auc1=all_auc.get("AUC@1px", 0.0),
        our_auc3=all_auc.get("AUC@3px", 0.0),
        our_auc5=all_auc.get("AUC@5px", 0.0),
        model_name=f"QSED-{D}D-Entropy+Euclid (n={n},y={y})",
        dim=D
    )
    results_text_lines.append("\n" + ripe_table + "\n")

    report_path = os.path.join(output_dir, "hpatches_results.txt")
    with open(report_path, "w", encoding="utf-8") as f:
        f.write("\n".join(results_text_lines))

    csv_path = os.path.join(output_dir, "hpatches_per_pair.csv")
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "sequence", "type", "pair", "num_kps_ref", "num_kps_tgt", "num_matches", "corner_error"
        ])
        writer.writeheader()
        for rec in per_pair_records:
            writer.writerow(rec)

    print(f"\n  Results saved to: {report_path}")
    print(f"  Per-pair CSV saved to: {csv_path}\n")
    print("\n".join(results_text_lines))

    return {
        "n": n, "y": y, "D": D,
        "augmented_dim": AUG_DIM,
        "diagnostics": aug_diagnostics,
        "elapsed": elapsed,
        "summary": summary_metrics,
        "ripe_table": ripe_table,
    }


In [6]:
# =============================================================================
# ONE-PAIR SMOKE TEST (AUGMENTED VECTORS)
# =============================================================================
import os
import cv2
import numpy as np

print("Running one-pair smoke test for Entropy + Euclidean pipeline...")

sample_seq = None
if os.path.isdir(dataset_dir):
    seqs = [d for d in sorted(os.listdir(dataset_dir)) if (d.startswith("i_") or d.startswith("v_")) and os.path.isfile(os.path.join(dataset_dir, d, "1.ppm"))]
    if len(seqs) > 0:
        sample_seq = os.path.join(dataset_dir, seqs[0])

if sample_seq is not None:
    print(f"Using HPatches sequence for smoke test: {os.path.basename(sample_seq)}")
    img_ref, scale_ref = load_hpatches_image(os.path.join(sample_seq, "1.ppm"), max_side=480)
    img_tgt, scale_tgt = load_hpatches_image(os.path.join(sample_seq, "2.ppm"), max_side=480)
    H_gt = load_homography(os.path.join(sample_seq, "H_1_2"))
    H_gt_scaled = scale_homography(H_gt, scale_ref, scale_tgt)
else:
    print("Generating synthetic image pair for smoke test...")
    img_ref = np.zeros((240, 320), dtype=np.float64)
    cv2.rectangle(img_ref, (40, 40), (120, 120), 200, -1)
    cv2.circle(img_ref, (200, 150), 30, 150, -1)
    cv2.putText(img_ref, "QSED", (60, 200), cv2.FONT_HERSHEY_SIMPLEX, 1, 255, 2)
    noise = np.random.RandomState(42).randint(0, 30, img_ref.shape)
    img_ref = np.clip(img_ref + noise, 0, 255)
    H_gt_scaled = np.array([[1.02, 0.02, 4.0], [-0.02, 0.98, -2.0], [0.0, 0.0, 1.0]], dtype=np.float64)
    img_tgt = cv2.warpPerspective(img_ref, H_gt_scaled, (320, 240))

# Test with (n=7, y=3, D=48, Augmented Width = 48 + 1024 = 1072)
n_smoke, y_smoke = 7, 3
D_smoke = n_smoke * n_smoke - 1
kps_ref = detect_default_keypoints(img_ref, max_keypoints=500)
kps_tgt = detect_default_keypoints(img_tgt, max_keypoints=500)
desc_ref = extract_all_entropy_descriptors(img_ref, kps_ref, norm="none", n=n_smoke, y=y_smoke)
desc_tgt = extract_all_entropy_descriptors(img_tgt, kps_tgt, norm="none", n=n_smoke, y=y_smoke)

C = compute_literal_score_matrix(desc_ref, desc_tgt)
aug_ref, mask_ref, aug_tgt, mask_tgt = build_augmented_vectors(desc_ref, desc_tgt, C, max_pad=1024)

# Verify shape and validity
assert aug_ref.shape == (len(kps_ref), D_smoke + 1024), f"Ref shape mismatch: {aug_ref.shape}"
assert aug_tgt.shape == (len(kps_tgt), D_smoke + 1024), f"Tgt shape mismatch: {aug_tgt.shape}"
assert np.all(aug_ref[:, D_smoke + len(kps_tgt):] == 0.0), "Padding non-zero!"
assert np.all(mask_ref[:, :len(kps_tgt)] == True) and np.all(mask_ref[:, len(kps_tgt):] == False)

print(f"Smoke test augmented shapes verified:")
print(f"  aug_ref shape: {aug_ref.shape} (valid entries: {len(kps_tgt)})")
print(f"  aug_tgt shape: {aug_tgt.shape} (valid entries: {len(kps_ref)})")

# Discard augmented vectors
del aug_ref, mask_ref, aug_tgt, mask_tgt

matches = match_mutual_nn_from_scores(C)
del C

H_est = estimate_homography_ransac(kps_ref, kps_tgt, matches, ransac_thresh=3.0)
assert H_est is not None, "RANSAC failed to estimate homography!"
corner_err = compute_corner_error(H_gt_scaled, H_est, img_ref.shape)

print(f"Smoke test successful!")
print(f"  Keypoints ref={len(kps_ref)}, tgt={len(kps_tgt)}")
print(f"  Reciprocal MNN matches: {len(matches)}")
print(f"  Corner reprojection error: {corner_err:.4f}px")


Running one-pair smoke test for Entropy + Euclidean pipeline...
Using HPatches sequence for smoke test: i_ajuntament
Smoke test augmented shapes verified:
  aug_ref shape: (500, 1072) (valid entries: 500)
  aug_tgt shape: (500, 1072) (valid entries: 500)
Smoke test successful!
  Keypoints ref=500, tgt=500
  Reciprocal MNN matches: 166
  Corner reprojection error: 0.4243px


In [7]:
# =============================================================================
# FULL HPATCHES BENCHMARK: EVALUATE ALL (n, y) CONFIGURATIONS
# =============================================================================
window_configs = [(7, 3), (7, 5), (9, 3), (9, 5)]
results_by_config = {}

for n, y in window_configs:
    d = n * n - 1
    output_dir = f"outputs/hpatches_entropy_plus_euclidean_n{n}_y{y}"
    print(f"\n{'='*75}\nExecuting Configuration: n={n}, y={y} (D={d}, Augmented={d+1024})\n{'='*75}")
    res = run_hpatches_configuration_entropy_plus_euclidean(
        dataset_dir=dataset_dir,
        output_dir=output_dir,
        n=n, y=y,
        max_keypoints=1024,
        max_side=480,
        ransac_thresh=3.0
    )
    results_by_config[(n, y)] = res



Executing Configuration: n=7, y=3 (D=48, Augmented=1072)
  QSED HPatches Evaluation — Entropy + Euclidean (n=7, y=3, D=48)
  Augmented Vector Width : 1072 (48 entropy + 1024 Euclidean)
  Normalization          : raw Von Neumann entropy bits (none)
  Matching               : Mutual Nearest Neighbor on literal score matrix C
  Augmented Vector Action: shape & validity inspection ONLY (not used for matching/RANSAC)
  Max Keypoints          : 1024
  Max Image Side         : 480px
  RANSAC Threshold       : 3.0px
  Dataset Directory      : data/hpatches-sequences-release
  Output Directory       : outputs/hpatches_entropy_plus_euclidean_n7_y3
  Found 116 sequences (57 illumination, 59 viewpoint)
  Total image pairs : 580



Eval n=7,y=3: 100%|██████████| 116/116 [27:08<00:00, 14.04s/seq]



  Results saved to: outputs/hpatches_entropy_plus_euclidean_n7_y3/hpatches_results.txt
  Per-pair CSV saved to: outputs/hpatches_entropy_plus_euclidean_n7_y3/hpatches_per_pair.csv

  QSED HPatches Benchmark Results (Entropy + Euclidean Augmented)
  Date             : 2026-10-30 19:11:16
  Configuration    : n=7, y=3 (D=48, Augmented Width=1072)
  Normalization    : raw Von Neumann entropy bits (none)
  Matching         : Mutual Nearest Neighbor on literal score matrix C
  Augmented Vector : shape (N, 1072) inspected and verified
  Max Keypoints    : 1024
  Max Image Side   : 480px
  RANSAC Threshold : 3.0px
  Total Time       : 1628.8s

  Augmented Vector Inspection Summary:
    Total Pairs Inspected         : 580
    Augmented Vector Dimensions   : (N_ref, 1072) and (N_tgt, 1072) [100% verified]
    Zero-Padding Integrity        : 100% verified exact zeros
    Validity Mask Integrity       : 100% verified clean distinction
    Ref Keypoints (valid lengths) : min=1024, mean=1024.0, ma

Eval n=7,y=5: 100%|██████████| 116/116 [33:10<00:00, 17.16s/seq]



  Results saved to: outputs/hpatches_entropy_plus_euclidean_n7_y5/hpatches_results.txt
  Per-pair CSV saved to: outputs/hpatches_entropy_plus_euclidean_n7_y5/hpatches_per_pair.csv

  QSED HPatches Benchmark Results (Entropy + Euclidean Augmented)
  Date             : 2026-10-30 19:44:27
  Configuration    : n=7, y=5 (D=48, Augmented Width=1072)
  Normalization    : raw Von Neumann entropy bits (none)
  Matching         : Mutual Nearest Neighbor on literal score matrix C
  Augmented Vector : shape (N, 1072) inspected and verified
  Max Keypoints    : 1024
  Max Image Side   : 480px
  RANSAC Threshold : 3.0px
  Total Time       : 1990.8s

  Augmented Vector Inspection Summary:
    Total Pairs Inspected         : 580
    Augmented Vector Dimensions   : (N_ref, 1072) and (N_tgt, 1072) [100% verified]
    Zero-Padding Integrity        : 100% verified exact zeros
    Validity Mask Integrity       : 100% verified clean distinction
    Ref Keypoints (valid lengths) : min=1024, mean=1024.0, ma

Eval n=9,y=3: 100%|██████████| 116/116 [41:03<00:00, 21.24s/seq]



  Results saved to: outputs/hpatches_entropy_plus_euclidean_n9_y3/hpatches_results.txt
  Per-pair CSV saved to: outputs/hpatches_entropy_plus_euclidean_n9_y3/hpatches_per_pair.csv

  QSED HPatches Benchmark Results (Entropy + Euclidean Augmented)
  Date             : 2026-10-30 20:25:31
  Configuration    : n=9, y=3 (D=80, Augmented Width=1104)
  Normalization    : raw Von Neumann entropy bits (none)
  Matching         : Mutual Nearest Neighbor on literal score matrix C
  Augmented Vector : shape (N, 1104) inspected and verified
  Max Keypoints    : 1024
  Max Image Side   : 480px
  RANSAC Threshold : 3.0px
  Total Time       : 2463.8s

  Augmented Vector Inspection Summary:
    Total Pairs Inspected         : 580
    Augmented Vector Dimensions   : (N_ref, 1104) and (N_tgt, 1104) [100% verified]
    Zero-Padding Integrity        : 100% verified exact zeros
    Validity Mask Integrity       : 100% verified clean distinction
    Ref Keypoints (valid lengths) : min=1024, mean=1024.0, ma

Eval n=9,y=5: 100%|██████████| 116/116 [50:54<00:00, 26.33s/seq]


  Results saved to: outputs/hpatches_entropy_plus_euclidean_n9_y5/hpatches_results.txt
  Per-pair CSV saved to: outputs/hpatches_entropy_plus_euclidean_n9_y5/hpatches_per_pair.csv

  QSED HPatches Benchmark Results (Entropy + Euclidean Augmented)
  Date             : 2026-10-30 21:16:26
  Configuration    : n=9, y=5 (D=80, Augmented Width=1104)
  Normalization    : raw Von Neumann entropy bits (none)
  Matching         : Mutual Nearest Neighbor on literal score matrix C
  Augmented Vector : shape (N, 1104) inspected and verified
  Max Keypoints    : 1024
  Max Image Side   : 480px
  RANSAC Threshold : 3.0px
  Total Time       : 3054.6s

  Augmented Vector Inspection Summary:
    Total Pairs Inspected         : 580
    Augmented Vector Dimensions   : (N_ref, 1104) and (N_tgt, 1104) [100% verified]
    Zero-Padding Integrity        : 100% verified exact zeros
    Validity Mask Integrity       : 100% verified clean distinction
    Ref Keypoints (valid lengths) : min=1024, mean=1024.0, ma

In [8]:
# =============================================================================
# CONSOLIDATED SUMMARY & AUGMENTED VECTOR DIAGNOSTICS
# =============================================================================
print("=" * 85)
print("   QSED ENTROPY + EUCLIDEAN: CONSOLIDATED RESULTS ACROSS CONFIGURATIONS")
print("=" * 85)
print(f"{'Config (n, y)':<15} | {'Aug Dim':<10} | {'AUC@1px':<10} | {'AUC@3px':<10} | {'AUC@5px':<10} | {'Failed H':<10} | {'Time':<8}")
print("-" * 85)

for (n, y), res in results_by_config.items():
    d = n * n - 1
    aug_dim = res["augmented_dim"]
    all_metrics = res["summary"].get("all", {})
    auc = all_metrics.get("auc", {})
    failed = all_metrics.get("failed", 0)
    total = all_metrics.get("total", 0)
    elapsed = res["elapsed"]
    print(f"n={n}, y={y:<8} | {aug_dim:<10} | {auc.get('AUC@1px', 0.0):>7.2f}%  | {auc.get('AUC@3px', 0.0):>7.2f}%  | {auc.get('AUC@5px', 0.0):>7.2f}%  | {failed}/{total:<5} | {elapsed:>6.1f}s")

print("=" * 85)
print("\nAugmented Vector Diagnostic Summary:")
for (n, y), res in results_by_config.items():
    diag = res["diagnostics"]
    print(f"  Config n={n}, y={y} -> Verified {diag['verified_ref_shapes']} ref and {diag['verified_tgt_shapes']} tgt augmented vectors of width {res['augmented_dim']}.")
    print(f"                       Padding zeros verified: 100%, Validity masks verified: 100%.")

print("\nBenchmark complete. HPatches was the only dataset evaluated (no Oxford, no MegaDepth).")


   QSED ENTROPY + EUCLIDEAN: CONSOLIDATED RESULTS ACROSS CONFIGURATIONS
Config (n, y)   | Aug Dim    | AUC@1px    | AUC@3px    | AUC@5px    | Failed H   | Time    
-------------------------------------------------------------------------------------
n=7, y=3        | 1072       |   23.31%  |   43.05%  |   50.36%  | 0/580   | 1628.8s
n=7, y=5        | 1072       |   18.21%  |   34.19%  |   40.24%  | 0/580   | 1990.8s
n=9, y=3        | 1104       |   23.74%  |   46.56%  |   55.30%  | 0/580   | 2463.8s
n=9, y=5        | 1104       |   19.88%  |   40.05%  |   47.91%  | 0/580   | 3054.6s

Augmented Vector Diagnostic Summary:
  Config n=7, y=3 -> Verified 580 ref and 580 tgt augmented vectors of width 1072.
                       Padding zeros verified: 100%, Validity masks verified: 100%.
  Config n=7, y=5 -> Verified 580 ref and 580 tgt augmented vectors of width 1072.
                       Padding zeros verified: 100%, Validity masks verified: 100%.
  Config n=9, y=3 -> Verified 580 ref 